In [193]:
import run as run
import chephy_model as cpm
import GNN_model
import gnn_utils
import numpy as np
import torch
import pandas as pd
import os
from torch_geometric.transforms import ToUndirected


In [194]:
atchley_path = "/home/dsi/chenbis/repos/sol_lab/files/atchley.csv"
atchley_df = pd.read_csv(atchley_path)
atchley_dict = atchley_df.set_index("amino.acid").to_dict(orient="index")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:

distance_threshold=0.01
use_mst=False
enrich=True

save_model_path = f"/home/dsi/chenbis/repos/sol_lab/tcr_epitope_prediction_gnn_mst/output/models/{distance_threshold if enrich else ''}_{'mst' if use_mst else ''}_{'enrich' if enrich else ''}"
model_type = "max"
os.makedirs(f"{save_model_path}/results/min", exist_ok=True)
os.makedirs(f"{save_model_path}/results/max", exist_ok=True)

In [196]:
def create_hetero_data(data, label_header, cdr3_header, atchley_dict, n=1000, distance_threshold = 0.2, use_mst=True, enrich=False):

    data = data.sample(n=n, random_state=42)

    data = run.prepare_data(data, cdr3_header, label_header)    
    data = cpm.truncate_sequences(data, cdr3_header, 4, 4)

    sequences = set(data["cdr3_truncated"])
    epitopes = list(set(data[label_header]))

    chephy_matrix, sequences_list = cpm.find_che_phy_dist(sequences)
    sequence_indices = np.arange(len(sequences_list))

    data['tcr_index'] = data['cdr3_truncated'].apply(lambda x: sequences_list.index(x) if x in sequences_list else -1)
    data['epitope_index'] = data[label_header].apply(lambda x: epitopes.index(x) if x in epitopes else -1)
    print("Done indexing")

    data_hetero, labels = gnn_utils.build_hetero_graph(
                            data,
                            chephy_matrix,
                            sequences_list,
                            epitopes,
                            atchley_dict,
                            distance_threshold=distance_threshold,
                            use_mst=use_mst,
                            enrich=enrich  
                        )
    
    data_hetero = ToUndirected()(data_hetero)
    return data_hetero, data


In [197]:
def generate_negative_edges(df, num_negatives, negative_score=0.0):
    """
    Efficiently generate negative cdr3~epitope pairs not present in df.
    """
    cdr3_pool = df['cdr3'].unique()
    epitope_pool = df['peptide'].unique()

    # Create full cartesian product
    all_combinations = pd.DataFrame(
        [(cdr3, epitope) for cdr3 in cdr3_pool for epitope in epitope_pool],
        columns=['cdr3', 'peptide']
    )

    # Remove positive (existing) pairs
    positives = df[['cdr3', 'peptide']].drop_duplicates()
    merged = all_combinations.merge(positives, on=['cdr3', 'peptide'], how='left', indicator=True)
    negatives = merged[merged['_merge'] == 'left_only'].drop(columns=['_merge'])

    # Shuffle and sample
    negatives = negatives.sample(n=min(num_negatives, len(negatives)), random_state=42).copy()
    negatives['score'] = negative_score
    negatives['source'] = 'NEGATIVE'

    # Truncate cdr3
    def truncate(seq):
        mid = (len(seq) + 1) // 2
        return seq[max(0, mid - 4):min(len(seq), mid + 4)]
    negatives['cdr3_truncated'] = negatives['cdr3'].apply(truncate)

    return negatives



def filter_data(df, source=None, min_score=None, max_score=None, num_negatives=0, negative_score=0.0):
    """
    Filters the combined DataFrame by source and score range, with optional negative edge generation.

    Parameters:
        df (pd.DataFrame): Combined dataset with 'source' and 'score' columns.
        source (str or list of str, optional): Filter by source(s) (e.g., "VDJDB" or ["McPAS", "TRAIT"]).
        min_score (float, optional): Minimum score (inclusive).
        max_score (float, optional): Maximum score (inclusive).
        num_negatives (int, optional): Number of negative cdr3~epitope pairs to generate.
        negative_score (float, optional): Score assigned to negative edges (default is 0.0).

    Returns:
        pd.DataFrame: Filtered DataFrame with optional negative edges appended.
    """
    filtered_df = df.copy()

    # Filter by source
    if source is not None:
        if isinstance(source, str):
            source = [source]
        filtered_df = filtered_df[filtered_df['source'].isin(source)]

    # Filter by score range
    if min_score is not None:
        filtered_df = filtered_df[filtered_df['score'] >= min_score]
    if max_score is not None:
        filtered_df = filtered_df[filtered_df['score'] <= max_score]

    # Efficient negative generation
    if num_negatives > 0:
        negative_df = generate_negative_edges(filtered_df, num_negatives, negative_score)
        filtered_df = pd.concat([filtered_df, negative_df], ignore_index=True)
    return filtered_df


In [198]:
def evaluate(df, n, cdr3_header = "cdr3", label_header = "peptide", distance_threshold=0.05, use_mst=True, enrich=False, model_type="min"):
    hetero_graph, data = create_hetero_data(df, label_header, cdr3_header, atchley_dict, n=len(df), distance_threshold = distance_threshold, use_mst=use_mst, enrich=enrich)

    # Create edge label indices for test data
    edge_pairs = data[['tcr_index', 'epitope_index']].values
    edge_label_index = torch.tensor(edge_pairs.T)

    # Create labels
    if "source" in data.columns:
        y = np.where(data['source'] == 'NEGATIVE', 0, 1)
    elif "Binding" in data.columns:
        y = np.where(data['Binding'] == 0, 0, 1)
    model = GNN_model.HeteroTCR(hetero_graph.metadata(), 256, 3, "SAGE")
    model = model.to(device)
    hetero_graph = hetero_graph.to(device)

    with torch.no_grad():  # Initialize lazy modules.
        out = model(hetero_graph.x_dict, hetero_graph.edge_index_dict, edge_label_index)

    val_model_path = save_model_path
    for root, dirs, files in os.walk(val_model_path):
        for file in files:
            if file.startswith(model_type):
                PATH = os.path.join(val_model_path, file)
                model.load_state_dict(torch.load(PATH))
                test_loss, test_binary_accuracy, test_ROCAUC, test_prob = GNN_model.predict(model, hetero_graph, edge_label_index, y)
                print("ACC: {:.4f}".format(test_binary_accuracy))
                print("AUC: {:.4f}".format(test_ROCAUC))

    # return hetero_graph, data, y, test_binary_accuracy, test_ROCAUC, test_prob
    return data, y, test_prob

In [199]:
def run_eval_and_save(data, save_file, cdr3_header = "cdr3", label_header = "peptide", distance_threshold=0.05, 
                      use_mst=True, enrich=False, model_type=model_type):

    data, y, test_prob = evaluate(data,
                                  len(data) , 
                                  cdr3_header = cdr3_header, 
                                  label_header = label_header, 
                                  distance_threshold=distance_threshold, 
                                 use_mst=use_mst, enrich=enrich, model_type=model_type)

    # Create a new dataframe with tcr, epitope, true label, and predicted binding probability
    results_df = pd.DataFrame({
        'tcr': data[cdr3_header],
        'epitope': data[label_header],
        'true_label': y,
        'predicted_binding_prob': test_prob.cpu().numpy()
    })

    results_df.to_csv(f"{save_model_path}/results/{model_type}/{save_file}", index=False)


In [200]:
data_combined = pd.read_csv('/home/dsi/chenbis/repos/sol_lab/files/combined_data.csv')
vdj0 = data_combined[(data_combined['source'] == "VDJDB") & (data_combined['score'] >= 0)]

In [201]:
vdj3 = filter_data(data_combined, source=["VDJDB"], num_negatives=1000, min_score=3, max_score=3)
vdj2 = filter_data(data_combined, source=["VDJDB"], num_negatives=3000, min_score=2, max_score=2)
vdj1 = filter_data(data_combined, source=["VDJDB"], num_negatives=10000, min_score=1, max_score=1)
vdj0_smaller = filter_data(vdj0.sample(n=10000), source=["VDJDB"], num_negatives=10000, min_score=0, max_score=0)

run_eval_and_save(vdj3, "vdj3.csv", distance_threshold = distance_threshold, use_mst=use_mst, enrich=enrich)
run_eval_and_save(vdj2, "vdj2.csv", distance_threshold = distance_threshold, use_mst=use_mst, enrich=enrich)
run_eval_and_save(vdj1, "vdj1.csv", distance_threshold = distance_threshold, use_mst=use_mst, enrich=enrich)
run_eval_and_save(vdj0_smaller, "vdj0.csv", distance_threshold = distance_threshold, use_mst=use_mst, enrich=enrich)

Done indexing


Using cache found in /home/dsi/chenbis/.cache/torch/hub/facebookresearch_esm_main
/tmp/ipykernel_2518711/1299457542.py:25: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  mode

ACC: 0.6361
AUC: 0.7169
Done indexing


Using cache found in /home/dsi/chenbis/.cache/torch/hub/facebookresearch_esm_main
/tmp/ipykernel_2518711/1299457542.py:25: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  mode

ACC: 0.6779
AUC: 0.6504
Done indexing


Using cache found in /home/dsi/chenbis/.cache/torch/hub/facebookresearch_esm_main
/tmp/ipykernel_2518711/1299457542.py:25: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  mode

ACC: 0.6601
AUC: 0.6656
Done indexing


Using cache found in /home/dsi/chenbis/.cache/torch/hub/facebookresearch_esm_main


ACC: 0.6283
AUC: 0.6508


/tmp/ipykernel_2518711/1299457542.py:25: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(PATH))


In [202]:
tsv1 = pd.read_csv("../data/train.tsv", sep="\t")
tsv2 = pd.read_csv("../data/test.tsv", sep="\t")
tsv_all = pd.concat([tsv1, tsv2], ignore_index=True)

tsv_cdr3_set = set(tsv_all["cdr3"])
tsv_peptide_set = set(tsv_all["peptide"])

cdr3_only_csv_vdj0 = vdj0[~vdj0["cdr3"].isin(tsv_cdr3_set)]
peptide_only_csv_vdj0 = vdj0[~vdj0["peptide"].isin(tsv_peptide_set)]
cdr3_and_peptide_only_csv_vdj0 = vdj0[
    (~vdj0["cdr3"].isin(tsv_cdr3_set)) &
    (~vdj0["peptide"].isin(tsv_peptide_set))
]


In [203]:
vdj0 = filter_data(vdj0.sample(n=20000), num_negatives=10000)
cdr3_only_csv_vdj0 = filter_data(cdr3_only_csv_vdj0.sample(n=20000), num_negatives=10000)
peptide_only_csv_vdj0 = filter_data(peptide_only_csv_vdj0, num_negatives=10000)
cdr3_and_peptide_only_csv_vdj0 = filter_data(cdr3_and_peptide_only_csv_vdj0, num_negatives=10000)

In [204]:
run_eval_and_save(vdj0, "none_excluded_vdj0.csv", distance_threshold = distance_threshold, use_mst=use_mst, enrich=enrich)
run_eval_and_save(cdr3_only_csv_vdj0, "cdr3_excluded_vdj0.csv", distance_threshold = distance_threshold, use_mst=use_mst, enrich=enrich)
run_eval_and_save(peptide_only_csv_vdj0, "peptide_excluded_vdj0.csv", distance_threshold = distance_threshold, use_mst=use_mst, enrich=enrich)
run_eval_and_save(cdr3_and_peptide_only_csv_vdj0, "cdr3_and_peptide_excluded_vdj0.csv", distance_threshold = distance_threshold, use_mst=use_mst, enrich=enrich)

Done indexing


Using cache found in /home/dsi/chenbis/.cache/torch/hub/facebookresearch_esm_main
/tmp/ipykernel_2518711/1299457542.py:25: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  mode

ACC: 0.6246
AUC: 0.6978
Done indexing


Using cache found in /home/dsi/chenbis/.cache/torch/hub/facebookresearch_esm_main
/tmp/ipykernel_2518711/1299457542.py:25: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  mode

ACC: 0.6143
AUC: 0.6889
Done indexing


Using cache found in /home/dsi/chenbis/.cache/torch/hub/facebookresearch_esm_main
/tmp/ipykernel_2518711/1299457542.py:25: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  mode

ACC: 0.5978
AUC: 0.6499
Done indexing


Using cache found in /home/dsi/chenbis/.cache/torch/hub/facebookresearch_esm_main


ACC: 0.5993
AUC: 0.6505


/tmp/ipykernel_2518711/1299457542.py:25: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(PATH))
